In [0]:
catalog="Ecommerce"

###***Order Items***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_order_items = spark.read.table(f"{catalog}.silver.slv_order_items_fact")

gld_order_items=gld_order_items.withColumn("gross_amount", col("unit_price")*col("quantity"))\
    .withColumn("discount_amount",col("gross_amount")*col("discount_pct")/100)\
    .withColumn("sales_amount",col("gross_amount")-col("discount_amount")+col("tax_amount"))\
    .withColumn("dateid",date_format(col("date"),"yyyyMMdd").cast(IntegerType()))\
    .withColumn("coupon_flag",when(col("coupon_code").isNotNull(),1).otherwise(0))



gld_order_items.select(col("dateid"),col("date"),col("order_ts").alias("transaction_ts"),col("Customer_id"),col("order_id"),col("item_seq"),col("product_id"),col("quantity"),col("unit_price_currency"),col("gross_amount"),col("unit_price"),col("discount_pct").alias("discount_percentage"),col("discount_amount"),col("sales_amount"),col("tax_amount"),col("channel"),col("coupon_code"),col("coupon_flag"),col("source_file"),col("ingested_at")).write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.Gold.gld_order_items_fact')


###***Order Returns***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_order_returns = spark.read.table(f"{catalog}.silver.slv_order_returns_fact")

gld_order_returns=gld_order_returns.withColumn("dateid",date_format(col("order_dt"),"yyyyMMdd").cast(IntegerType()))

gld_order_returns.select(col("dateid"),col("order_dt"),col("return_ts").alias("transaction_ts"),col("order_id"),col("reason").alias("reason_of_returns"),col("source_file"),col("ingested_at")).write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.Gold.gld_order_returns_fact')


###***Order Shipments***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_order_shipments = spark.read.table(f"{catalog}.silver.slv_order_shipments_fact")

gld_order_shipments=gld_order_shipments.withColumn("dateid",date_format(col("order_dt"),"yyyyMMdd").cast(IntegerType())).withColumnRenamed("carrier","Delivery_partner")

gld_order_shipments.select(col("dateid"),col("order_dt"),col("shipment_id"),col("order_id"),col("Delivery_partner"),col("source_file"),col("ingested_at")).write.format("delta").mode("overwrite").option("mergeSchema",True).saveAsTable(f"{catalog}.Gold.gld_order_shipments_fact")